In [0]:
from pyspark.sql.functions import col, avg
from pyspark.sql.window import Window

# 1. Leer de la capa Silver
df_silver = spark.table("silver_nvda_prices")

# 2. Definir "Ventanas" de tiempo
# Esto le dice a Spark: "Mira hacia atrás X filas ordenadas por fecha"
window_7 = Window.orderBy("date").rowsBetween(-6, 0)
window_30 = Window.orderBy("date").rowsBetween(-29, 0)

# 3. Calcular las medias móviles (SMA)
df_features = df_silver.withColumn("sma_7", avg(col("close_price")).over(window_7)) \
                       .withColumn("sma_30", avg(col("close_price")).over(window_30))

# 4. Mostrar el resultado
display(df_features.select("date", "close_price", "sma_7", "sma_30").limit(10))

In [0]:
from pyspark.sql.functions import lead, when

# 1. Creamos el TARGET: El precio de cierre del día siguiente
# lead(col, 1) trae el valor de la siguiente fila
window_spec = Window.orderBy("date")
df_features = df_features.withColumn("next_day_close", lead(col("close_price"), 1).over(window_spec))

# 2. Creamos la etiqueta binaria: 1 si sube, 0 si baja o se mantiene
df_features = df_features.withColumn(
    "target", 
    when(col("next_day_close") > col("close_price"), 1).otherwise(0)
)

# 3. Calculamos Volatilidad diaria (High - Low)
df_features = df_features.withColumn("daily_volatility", col("high_price") - col("low_price"))

# 4. Limpieza final: Quitamos la última fila (porque no tiene "día siguiente") y nulos de las medias
df_final_features = df_features.filter(col("next_day_close").isNotNull()) \
                               .filter(col("sma_30").isNotNull())

# Mostramos el resultado final
display(df_final_features.select("date", "close_price", "sma_7", "target", "daily_volatility").limit(10))

In [0]:
# Guardamos en la Capa Gold
# Esta tabla es la que "alimentará" al modelo de ML
df_final_features.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_nvda_features")

print("¡Capa GOLD finalizada! Tabla 'gold_nvda_features' lista para el entrenamiento.")